In [51]:
%%HTML
<style>
    body {
        --vscode-font-family: "ComicSansMS"
    }
</style>

## Design Excercise 1: Design an LDO in spice using the functions


In [52]:
## Target Specs


I_load = 10e-3
I_res_div = (1/100) * I_load
pmos_pass_char_csv = 'ldo_pmos_kgm_char_Msweep_10mA.csv'
V_out = 1.2
V_ctrl = 0.9
V_droop_max = 100e-3
C_load = 10e-9
PSRR_target_dB = -32
PM_target = 60
kgm_max = 25

M_scale_opt = 100

In [53]:

t_response = (C_load * V_droop_max) / I_load  # Response Time in seconds
print(f"Response Time: {t_response * 1e9:.2f} ns")


Response Time: 100.00 ns


In [54]:
csv_file_pmos = 'ldo_pmos_kgm_char_Msweep_10mA.csv'
csv_file_nmos = "sf_nmos_char_vg_1_3.csv"

In [55]:
C_L = C_load  # Load Capacitance in Farads
PSRR_dB = PSRR_target_dB  # Power Supply Rejection Ratio in dB
I_max = I_load  # Maximum Current in Amperes
beta = I_load/(I_load+I_res_div)  # Beta ratio
PSRR = 10 ** (PSRR_dB / 20)  # Convert PSRR from dB to linear scale

## Assuming response is dominated due to Loop gain bandwidth 
f_bw = PSRR /t_response  # Bandwidth in Hz
print(f"Estimated Bandwidth (f_bw): {f_bw / 1e6:.2f} MHz")

## Estimating I_q using functions defined 
from LDO_functions import estimate_ldo_power_and_pass_params

result_ldo = estimate_ldo_power_and_pass_params(
    PSRR_target_dB=PSRR_target_dB,
    f_bw=f_bw,
    PM_target=PM_target,
    I_load=I_load,
    I_res_div=I_res_div,
    V_out=V_out,
    V_ctrl=V_ctrl,
    C_out=C_L,
    csv_file_nmos=csv_file_nmos,
    csv_file_pmos=csv_file_pmos,
    kgm_target=kgm_max
)

print("LDO Power and Pass Parameters Estimation:")
for key, value in result_ldo.items():
    print(f"{key}: {value}")

I_q = result_ldo['I_total']  # Quiescent Current in Amperes


Estimated Bandwidth (f_bw): 0.25 MHz
Resistor Divider Resistance (R_res_div) for 1% current overhead at I_max: 12000.00 Ohms
Total current through the PMOS pass transistor (I_total): 10.10 mA
Gate-Drain Capacitance (Cgd) at I_total 10.10 mA: 0.390 pF
Gate-Source Capacitance (Cgs) at I_total 10.10 mA: 0.958 pF
Effective Load Resistance (R_L_eff) considering Rds: 287.09 Ohms
Effective Load Capacitance at I_min: 10000.406 pF
Load Resistance (R_L) required for bandwidth target 251188.6 Hz: 63.36 Ohms
Using Resistor Divider Resistance (R_res_div): 12000.00 Ohms
Av_pass at I_total 10.10 mA: 24.53 V/V
Stage 2 required current (I_req_stage2): 0.0950 mA
Stage 2 input capacitance (C_in_stage2): 90.76 pF
Stage 1 required current (I_req_stage1): 0.8389 mA
Total required current (I_req_total): 0.9339 mA
LDO Power and Pass Parameters Estimation:
C_in_pass: 1.0916659361100337e-11
Av_pass: 24.533791468811994
k_res_div: 0.75
details_pass: {'idx_both': 37, 'M_scale_both': 888.6238, 'gm_p_both': 0.085456